In [1]:
import pandas as pd
import category_encoders as ce
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "basic_clean.csv"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"

df = pd.read_csv(DATA_PATH, low_memory = False)

df.shape

(99340, 51)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99340 entries, 0 to 99339
Data columns (total 51 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   encounter_id              99340 non-null  int64 
 1   patient_nbr               99340 non-null  int64 
 2   race                      97108 non-null  object
 3   gender                    99340 non-null  object
 4   age                       99340 non-null  object
 5   weight                    3125 non-null   object
 6   admission_type_id         99340 non-null  int64 
 7   discharge_disposition_id  99340 non-null  int64 
 8   admission_source_id       99340 non-null  int64 
 9   time_in_hospital          99340 non-null  int64 
 10  payer_code                59943 non-null  object
 11  medical_specialty         50726 non-null  object
 12  num_lab_procedures        99340 non-null  int64 
 13  num_procedures            99340 non-null  int64 
 14  num_medications       

Odvajamo ciljnu promenljivu od atributa. <br>
Kolone readmitted i target_readmitted_30 uklanjamo iz skupa atributa kako bismo sprecili data leakage.

In [4]:
y = df["target_readmitted_30"]

X = df.drop(columns=["readmitted", "target_readmitted_30"])

X.shape, y.shape

((99340, 49), (99340,))

In [5]:
columns_to_drop = [
    "encounter_id",
    "patient_nbr",
    "weight",
    "payer_code"
]

X_model = X.drop(columns=columns_to_drop)

X_model.shape

(99340, 45)

Izbacujemo id kolone jer ne predstavljaju osobine pacijenta koje bi model trebalo da koristi za predvidjanje i kolonu weight zbog visokog procenta missing vrednosti. (96.86%)<br>
Medicinske podatke zbog visokog broja nedostajucih vrednosti necemo izbacivati jer cinjenica da nesto nije mereno moze biti signal, NaN cemo tretirati kao posebnu, Missing kategoriju.

In [6]:
categorical_id_columns = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

Zbog prethodnog mapiranja, ove id kolone mozemo tretirati kao kategorije.

In [7]:
numeric_features = (
    X_model
    .select_dtypes(include=["int64", "float64"])
    .columns
    .difference(categorical_id_columns)
    .tolist()
)

categorical_features = (
    X_model
    .select_dtypes(include=["object"])
    .columns
    .tolist()
    + categorical_id_columns
)

print("Numeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numeric features: 8
['num_lab_procedures', 'num_medications', 'num_procedures', 'number_diagnoses', 'number_emergency', 'number_inpatient', 'number_outpatient', 'time_in_hospital']

Categorical features: 37
['race', 'gender', 'age', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id']


Provera da li smo propustili neku kolonu.

In [8]:
selected_features = numeric_features + categorical_features

missing_from_selection = set(X_model.columns) - set(selected_features)
extra_in_selection = set(selected_features) - set(X_model.columns)

missing_from_selection, extra_in_selection

(set(), set())

---

Podatke delimo na train, validation i test skup podataka, u odnosu 60, 20, 20. <br>
train skup koristimo za obucavanje modela, validacioni za biranje, poredjenje modela i podesavanje hiperparametara, dok testni skup sluzi za finalnu, nepristrasnu procenu performansi modela.

In [9]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_model,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=42,
    stratify=y_train_val
)

X_train.shape, X_val.shape, X_test.shape

((59604, 45), (19868, 45), (19868, 45))

Proveravamo raspodelu targeta i pravimo tabelu za bolji prikaz.

In [10]:
split_distribution = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(y_train), len(y_val), len(y_test)],
    "positive_count": [
        y_train.sum(),
        y_val.sum(),
        y_test.sum()
    ],
    "positive_rate_percent": [
        round(y_train.mean() * 100, 2),
        round(y_val.mean() * 100, 2),
        round(y_test.mean() * 100, 2)
    ]
})

split_distribution.to_csv(TABLES_DIR / "split_distribution.csv", index=False)

split_distribution

,split,rows,positive_count,positive_rate_percent
0,train,59604,6788,11.39
1,validation,19868,2263,11.39
2,test,19868,2263,11.39


Na osnovu tabele iz prethodne faze, columns_summary, mozemo bolje da razmislimo o encoding strategiji. <br>
Radi preglednosti, pravimo jos jednu tabelu.

In [11]:
categorical_cardinality = pd.DataFrame({
    "feature": categorical_features,
    "unique_values": [X_model[col].nunique(dropna=True) for col in categorical_features],
    "missing_count": [X_model[col].isna().sum() for col in categorical_features],
    "missing_percent": [
        round(X_model[col].isna().sum() / len(X_model) * 100, 2)
        for col in categorical_features
    ]
}).sort_values("unique_values", ascending=False)

categorical_cardinality.to_csv(
    TABLES_DIR / "categorical_cardinality.csv",
    index=False
)

categorical_cardinality

,feature,unique_values,missing_count,missing_percent
6,diag_3,786,1419,1.43
5,diag_2,747,356,0.36
4,diag_1,715,20,0.02
3,medical_specialty,72,48614,48.94
35,discharge_disposition_id,21,0,0.00
36,admission_source_id,17,0,0.00
2,age,10,0,0.00
34,admission_type_id,8,0,0.00
0,race,5,2232,2.25
27,glyburide-metformin,4,0,0.00


Sve atribute kardinalnosti <=72, cemo obraditi one-hot encoding metodom.<br>
Kao sto je i ranije zapazeno, A1Cresult i max_glu_serum, imaju dosta nedostajucih vrednosti, sto ce ciniti posebnu kategoriju. <br>
Za diag 3, 2 i 1 se nece koristiti klasicna one-hot encoding metoda zbog prevelikog broja kategorija, sto moze napraviti veliku i retku matricu. <br>
U razmatranju su target encoding, frequency encoding, i grupisanje retkih dijagnoza + one-hot encoding. <br>
Target encoding dobro hvata vezu feature-a sa targetom, to je zapravo stopa readmisije po kategoriji, ali je velika opasnost od data leakage-a. Ako se svaka kategorija enkoduje prosekom targeta iz istih redova na kojima se model trenira, svaki red indirektno ucestvuje u izracunavanju svoje encoded vrednosti, sto dovodi do overfitting-a. <br>
Mozemo dodati smoothing parametar, ili probati cross validation pristup. <br>
Frequency encoding enkoduje kategorije po broju pojavljivanja (% pojavljivanja), govori nam da li je dijagnoza cesta ali retka, ali ne koristi vezu kategorije sa targetom, odnosno ne govori da li dijagnoza ima vecu ili manju stopu readmisije. Bezbedniji je za target leakage.<br>
I grupisanje retkih + one-hot znaci da umesto za svaku kategoriju pravimo posebnu one-hot kolonu, zadrzacemo samo one koje se pojavljuju dovoljno cesto, a sve retke spojiti u jednu kategoriju. Gubimo razliku izmedju retkih dijagnoza ali je to u redu jer model nema dovoljno primera da nauci efekte svake.

In [12]:
diagnosis_columns = [
    "diag_1",
    "diag_2",
    "diag_3"
]

one_hot_columns = [
    col for col in categorical_features
    if col not in diagnosis_columns
]

print("One-hot columns:", len(one_hot_columns))
print(one_hot_columns)

print("\nDiagnosis columns:", diagnosis_columns)

One-hot columns: 34
['race', 'gender', 'age', 'medical_specialty', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id']

Diagnosis columns: ['diag_1', 'diag_2', 'diag_3']


In [13]:
preprocessing_feature_groups = pd.DataFrame({
    "feature": numeric_features + one_hot_columns + diagnosis_columns,
    "group": (
        ["numeric"] * len(numeric_features)
        + ["one_hot_categorical"] * len(one_hot_columns)
        + ["high_cardinality_diagnosis"] * len(diagnosis_columns)
    )
})

preprocessing_feature_groups.to_csv(
    TABLES_DIR / "preprocessing_feature_groups.csv",
    index=False
)

preprocessing_feature_groups

,feature,group
0,num_lab_procedures,numeric
1,num_medications,numeric
2,num_procedures,numeric
3,number_diagnoses,numeric
4,number_emergency,numeric
5,number_inpatient,numeric
6,number_outpatient,numeric
7,time_in_hospital,numeric
8,race,one_hot_categorical
9,gender,one_hot_categorical


---

Potrebna je instalacija category_encoders biblioteke, pip install category_encoders

Nedostajuce vrednosti numerickih kolona popunjavamo medijanom zato sto je otpornija na ekstremne vrednosti od proseka. <br>
Standardizujemo podatke kako bi svi bili izrazeni na istoj skali.<br>
Nedostajuce vrednosti kategorickih kolona popunjavamo tako sto ih zamenjujemo kategorijom Missing zato sto neka nedostajuca kategorija moze biti informativna. <br>

Nove kategorije koje se pojave u validacionom ili test skupu a nisu postojale u train-u se tretiraju kao retke (handle_unknown). <br>
Retke su one koje se pojavljuju manje od 50 puta, grupisu se zajedno.

In [14]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

one_hot_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=50
    ))
])

---

## Grupisanje retkih + one hot

In [15]:
preprocessor_ohe_rare_diag = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", one_hot_transformer, categorical_features)
    ]
)

## Frequency encoding za dijagnoze

In [16]:
diagnosis_count_transformer = Pipeline(steps=[
    ("count_encoder", ce.CountEncoder(
        cols=diagnosis_columns,
        normalize=True,
        handle_missing="value",
        handle_unknown=0
    )),
    ("scaler", StandardScaler())
])

preprocessor_frequency_diag = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("onehot", one_hot_transformer, one_hot_columns),
        ("diag_frequency", diagnosis_count_transformer, diagnosis_columns)
    ]
)

## Target encoding za dijagnoze

In [17]:
diagnosis_target_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("target_encoder", TargetEncoder(
        target_type="binary",
        smooth="auto",
        cv=5
    )),
    ("scaler", StandardScaler())
])

preprocessor_target_diag = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("onehot", one_hot_transformer, one_hot_columns),
        ("diag_target", diagnosis_target_transformer, diagnosis_columns)
    ]
)

smoothing automatski prilagodjava retke kategorije globalnom proseku. <br>
5 cv, da red ne bi ucestvovao u racunanju svoje sopstvene encoded vrednosti, trening skup se podeli na 5 delova, za svaki deo vrednosti se racunaju koristeci ostala 4.

Jedan dictionary kako bismo uporedili sve tehnike.

In [18]:
preprocessing_experiments = {
    "ohe_rare_diag": preprocessor_ohe_rare_diag,
    "frequency_diag": preprocessor_frequency_diag,
    "target_diag": preprocessor_target_diag
}

In [19]:
preprocessing_results = []

for experiment_name, preprocessor in preprocessing_experiments.items():
    X_train_processed = preprocessor.fit_transform(
        X_train,
        y_train
    )

    X_val_processed = preprocessor.transform(X_val)

    preprocessing_results.append({
        "experiment": experiment_name,
        "train_shape": X_train_processed.shape,
        "validation_shape": X_val_processed.shape
    })

preprocessing_results = pd.DataFrame(
    preprocessing_results
)

preprocessing_results

,experiment,train_shape,validation_shape
0,ohe_rare_diag,"(59604, 624)","(19868, 624)"
1,frequency_diag,"(59604, 168)","(19868, 168)"
2,target_diag,"(59604, 168)","(19868, 168)"


---

Baseline model za odabir najbolje strategije.

In [20]:
baseline_results = []

for experiment_name, preprocessor in preprocessing_experiments.items():
    model_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42
        ))
    ])

    model_pipeline.fit(X_train, y_train)

    y_val_pred = model_pipeline.predict(X_val)
    y_val_proba = model_pipeline.predict_proba(X_val)[:, 1]

    baseline_results.append({
        "preprocessing_strategy": experiment_name,
        "accuracy": round(accuracy_score(y_val, y_val_pred), 4),
        "precision": round(precision_score(y_val, y_val_pred), 4),
        "recall": round(recall_score(y_val, y_val_pred), 4),
        "f1": round(f1_score(y_val, y_val_pred), 4),
        "roc_auc": round(roc_auc_score(y_val, y_val_proba), 4),
    })

baseline_results = pd.DataFrame(baseline_results)
baseline_results

,preprocessing_strategy,accuracy,precision,recall,f1,roc_auc
0,ohe_rare_diag,0.6561,0.1805,0.5705,0.2742,0.6626
1,frequency_diag,0.6675,0.1837,0.5572,0.2763,0.6673
2,target_diag,0.6628,0.1842,0.5718,0.2786,0.6719


In [21]:
baseline_results.to_csv(
    TABLES_DIR / "baseline_preprocessing_comparison.csv",
    index=False
)

Na osnovu  f1 i roc auc vrednosti, biramo target encoding metodu.

Primenicemo preprocessing strategiju i sacuvati python objekte kako bismo koristili spremljene podatke nadalje u projektu.

In [22]:
MODELING_DATA_DIR = PROJECT_ROOT / "data" / "processed" / "modeling"
MODELING_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [23]:
final_preprocessor = preprocessor_target_diag

X_train_processed = final_preprocessor.fit_transform(X_train, y_train)
X_val_processed = final_preprocessor.transform(X_val)
X_test_processed = final_preprocessor.transform(X_test)

X_train_processed.shape, X_val_processed.shape, X_test_processed.shape

((59604, 168), (19868, 168), (19868, 168))

In [24]:
joblib.dump(X_train_processed, MODELING_DATA_DIR / "X_train_target_diag.joblib")
joblib.dump(X_val_processed, MODELING_DATA_DIR / "X_val_target_diag.joblib")
joblib.dump(X_test_processed, MODELING_DATA_DIR / "X_test_target_diag.joblib")

joblib.dump(y_train.to_numpy(), MODELING_DATA_DIR / "y_train.joblib")
joblib.dump(y_val.to_numpy(), MODELING_DATA_DIR / "y_val.joblib")
joblib.dump(y_test.to_numpy(), MODELING_DATA_DIR / "y_test.joblib")

joblib.dump(final_preprocessor, MODELING_DATA_DIR / "target_diag_preprocessor.joblib")

['C:\\Users\\user\\Documents\\MasterRad\\data\\processed\\modeling\\target_diag_preprocessor.joblib']

Cuvamo imena feature-a, kako bismo kasnije gledali feature importance i lakse interpretirali model.

In [25]:
try:
    feature_names = final_preprocessor.get_feature_names_out()

    feature_names_df = pd.DataFrame({
        "feature_name": feature_names
    })

    feature_names_df.to_csv(
        MODELING_DATA_DIR / "target_diag_feature_names.csv",
        index=False
    )

    feature_names_df.head()
except Exception as e:
    print("Feature names could not be extracted:")
    print(e)

In [26]:
feature_names_path = (
    MODELING_DATA_DIR /
    "target_diag_feature_names.csv"
)

print(feature_names_path.exists())

True


In [27]:
print(
    len(feature_names),
    X_train_processed.shape[1]
)

168 168
